In [ ]:
from pathlib import Path
from itertools import product
import pandas as pd
from IPython.display import display

from lenskit.data import from_interactions_df, ItemListCollection
from lenskit.splitting import sample_users, SampleN
from lenskit.basic import BiasScorer, PopScorer
from lenskit.knn import ItemKNNScorer, ItemKNNConfig
from lenskit.pipeline import predict_pipeline, topn_pipeline
from lenskit.batch import predict, recommend
from lenskit.metrics import RMSE, MAE, Precision, Recall, NDCG, RunAnalysis

SEED = 42
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'data'/'ratings.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run from the repository root or its notebooks directory.')
DATA = ROOT/'data'

ratings = pd.read_csv(DATA/'ratings.csv')
books = pd.read_csv(DATA/'books.csv')
print(f'Loaded {len(ratings):,} ratings, {ratings.user_id.nunique():,} users, {ratings.book_id.nunique():,} books')

ds = from_interactions_df(ratings, user_col='user_id', item_col='book_id', rating_col='rating')

Loaded 5,976,479 ratings, 53,424 users, 10,000 books


In [ ]:
N_TEST_USERS = 5000
N_VAL_USERS = 2000
HOLDOUT_PER_USER = 5

test_split = sample_users(ds, N_TEST_USERS, SampleN(HOLDOUT_PER_USER, rng=SEED), rng=SEED)
print(f'Test split: {test_split.train.user_count:,} train users, {test_split.test_size:,} held-out test ratings')

# Carve a validation split out of the *training* portion only — test stays untouched until final evaluation
val_split = sample_users(test_split.train, N_VAL_USERS, SampleN(HOLDOUT_PER_USER, rng=SEED + 1), rng=SEED + 1)
print(f'Validation split: {val_split.train.user_count:,} train users, {val_split.test_size:,} held-out val ratings')

# Relevance for ranking metrics: held-out rating >= 4
def relevant_only(test_collection):
    df = test_collection.to_df()
    return ItemListCollection.from_df(df[df.rating >= 4], test_collection.key_type)

test_relevant = relevant_only(test_split.test)
val_relevant = relevant_only(val_split.test)
print(f'Test: {test_relevant.total_items():,} / {test_split.test.total_items():,} held-out ratings are relevant (>=4)')

Test split: 53,424 train users, 25,000 held-out test ratings
Validation split: 53,424 train users, 10,000 held-out val ratings
Test: 17,218 / 25,000 held-out ratings are relevant (>=4)


In [ ]:
rmse_mae = RunAnalysis()
rmse_mae.add_metric(RMSE())
rmse_mae.add_metric(MAE())

ranking_metrics = RunAnalysis()
ranking_metrics.add_metric(Precision(k=10))
ranking_metrics.add_metric(Recall(k=10))
ranking_metrics.add_metric(NDCG(k=10))

# --- Bias baseline: rating prediction ---
bias_pred_pipe = predict_pipeline(BiasScorer(), fallback=False)
bias_pred_pipe.train(test_split.train)
bias_preds = predict(bias_pred_pipe, test_split.test)
bias_rating_metrics = rmse_mae.measure(bias_preds, test_split.test).list_summary()
display(bias_rating_metrics)

# --- Popularity baseline: top-N ranking ---
pop_topn_pipe = topn_pipeline(PopScorer(), n=10)
pop_topn_pipe.train(test_split.train)
pop_recs = recommend(pop_topn_pipe, test_split.test_requests, n=10)
pop_ranking_metrics = ranking_metrics.measure(pop_recs, test_relevant).list_summary()
display(pop_ranking_metrics)

,mean,median,std
metric,,,
RMSE,0.790667,0.751035,0.326482
MAE,0.668259,0.634679,0.280767


C:\Users\EGE\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lenskit\metrics\ranking\_pr.py:71: RuntimeWarning: invalid value encountered in scalar divide
  return ngood / nrel
C:\Users\EGE\AppData\Local\Temp\ipykernel_26908\698166165.py:21: DataWarning: test item list is empty
  pop_ranking_metrics = ranking_metrics.measure(pop_recs, test_relevant).list_summary()


2026-11-11 11:58:02 [warning  ] could not find test data for 92 lists


,mean,median,std
metric,,,
Precision@10,0.019100,0.0,0.045581
Recall@10,0.056714,0.0,0.142638
NDCG@10,0.042540,0.0,0.112673


In [ ]:
MAX_NBRS_GRID = [10, 20, 40, 80]
MIN_NBRS_GRID = [1, 3, 5]

tuning_rows = []
for max_nbrs, min_nbrs in product(MAX_NBRS_GRID, MIN_NBRS_GRID):
    cfg = ItemKNNConfig(max_nbrs=max_nbrs, min_nbrs=min_nbrs, feedback='explicit')
    pipe = predict_pipeline(ItemKNNScorer(cfg), fallback=BiasScorer())
    pipe.train(val_split.train)
    preds = predict(pipe, val_split.test)
    summary = rmse_mae.measure(preds, val_split.test).list_summary()
    tuning_rows.append({
        'max_nbrs': max_nbrs, 'min_nbrs': min_nbrs,
        'val_rmse': summary.loc['RMSE', 'mean'],
        'val_mae': summary.loc['MAE', 'mean'],
    })

tuning_results = pd.DataFrame(tuning_rows).sort_values('val_rmse')
display(tuning_results)

best = tuning_results.iloc[0]
BEST_MAX_NBRS, BEST_MIN_NBRS = int(best.max_nbrs), int(best.min_nbrs)
print(f'Best config: max_nbrs={BEST_MAX_NBRS}, min_nbrs={BEST_MIN_NBRS} (val RMSE={best.val_rmse:.4f})')

KeyboardInterrupt: 

In [ ]:
final_cfg = ItemKNNConfig(max_nbrs=BEST_MAX_NBRS, min_nbrs=BEST_MIN_NBRS, feedback='explicit')
item_knn = ItemKNNScorer(final_cfg)

# Rating prediction, with Bias fallback (used for any item-kNN can't score)
iknn_pred_pipe = predict_pipeline(item_knn, fallback=BiasScorer())
iknn_pred_pipe.train(test_split.train)
iknn_preds = predict(iknn_pred_pipe, test_split.test)
iknn_rating_metrics = rmse_mae.measure(iknn_preds, test_split.test).list_summary()

# Top-N ranking
iknn_topn_pipe = topn_pipeline(item_knn, n=10)
iknn_topn_pipe.train(test_split.train)
iknn_recs = recommend(iknn_topn_pipe, test_split.test_requests, n=10)
iknn_ranking_metrics = ranking_metrics.measure(iknn_recs, test_relevant).list_summary()

comparison_rating = pd.DataFrame({
    'Bias baseline': bias_rating_metrics['mean'],
    'Item-kNN (tuned)': iknn_rating_metrics['mean'],
})
comparison_ranking = pd.DataFrame({
    'Popularity baseline': pop_ranking_metrics['mean'],
    'Item-kNN (tuned)': iknn_ranking_metrics['mean'],
})

print('Rating prediction (lower is better):')
display(comparison_rating)
print('Top-10 ranking (higher is better):')
display(comparison_ranking)

In [ ]:
# Same config, but WITHOUT a fallback: items ItemKNN cannot score come back as missing
iknn_nofallback_pipe = predict_pipeline(item_knn, fallback=False)
iknn_nofallback_pipe.train(test_split.train)
iknn_nofallback_preds = predict(iknn_nofallback_pipe, test_split.test)

nf_df = iknn_nofallback_preds.to_df()
scored = nf_df.dropna(subset=['score']) if 'score' in nf_df.columns else nf_df
total_requested = test_split.test.total_items()
n_covered = len(scored)

print(f'Total held-out (user, item) test pairs: {total_requested:,}')
print(f'Item-kNN produced a direct score for: {n_covered:,} ({n_covered/total_requested*100:.2f}%)')
print(f'Required Bias fallback for: {total_requested - n_covered:,} ({(total_requested - n_covered)/total_requested*100:.2f}%)')